# EDA: Cleaning Steps for Home Credit Default Risk

This notebook explores the application data and documents the **cleaning and derived-feature steps** that are implemented in `src/feature_engineering.py`. The EDA leads to the same decisions already applied there.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = pathlib.Path(".").resolve()
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Load application data

We use `application_train.csv` (same as in `feature_engineering.build_full_dataset`).

In [ ]:
app_path = DATA_DIR / "application_train.csv"
if not app_path.exists():
    app_path = DATA_DIR / "application_test.csv"  # fallback
df = pd.read_csv(app_path)
print(f"Shape: {df.shape}")
df.head()

## 2. EDA: DAYS_EMPLOYED — sentinel value 365243

**Question:** Are there impossible or placeholder values in employment length?

`DAYS_EMPLOYED` = "How many days before the application the person started current employment". So it should be negative (past) or possibly zero. A large *positive* value would mean "started employment in the future", which is a known **sentinel for unemployed/unknown** in this dataset.

In [ ]:
de = df["DAYS_EMPLOYED"]
print("DAYS_EMPLOYED summary:")
print(de.describe())
print(f"\nUnique values (sample): {sorted(de.dropna().unique())[:10]} ... {sorted(de.dropna().unique())[-5:]}")
print(f"\nValue 365243 count: {(de == 365243).sum()}")
print(f"365243 as share of non-null: {(de == 365243).sum() / de.notna().sum() * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
de.hist(bins=50, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_title("DAYS_EMPLOYED (raw)")
axes[0].set_xlabel("DAYS_EMPLOYED")
# Exclude 365243 to see the rest of the distribution
de_clean = de.replace(365243, np.nan)
de_clean.dropna().hist(bins=50, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_title("DAYS_EMPLOYED (excluding 365243)")
axes[1].set_xlabel("DAYS_EMPLOYED")
plt.tight_layout()
plt.show()

**Conclusion:** `365243` is a sentinel (≈ 1000 years in the future). It should be treated as **missing** so that:
- we don't use it in means/medians,
- derived features like employment years are not distorted.

**Cleaning step (as in `feature_engineering.clean_application`):** `df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)`

## 3. EDA: Columns used for derived features and division by zero

The pipeline builds:
- **AGE_YEARS** = -DAYS_BIRTH / 365.25  (no division by zero)
- **EMPLOYMENT_YEARS** = -DAYS_EMPLOYED / 365.25  (after replacing 365243)
- **INCOME_TO_CREDIT_RATIO** = AMT_INCOME_TOTAL / AMT_CREDIT
- **ANNUITY_TO_INCOME_RATIO** = AMT_ANNUITY / AMT_INCOME_TOTAL
- **CREDIT_TO_GOODS_RATIO** = AMT_CREDIT / AMT_GOODS_PRICE
- **INCOME_PER_FAMILY_MEMBER** = AMT_INCOME_TOTAL / CNT_FAM_MEMBERS

Check denominators for zeros.

In [ ]:
denominators = {
    "AMT_CREDIT": "INCOME_TO_CREDIT_RATIO",
    "AMT_INCOME_TOTAL": "ANNUITY_TO_INCOME_RATIO",
    "AMT_GOODS_PRICE": "CREDIT_TO_GOODS_RATIO",
    "CNT_FAM_MEMBERS": "INCOME_PER_FAMILY_MEMBER",
}
for col, feat in denominators.items():
    zeros = (df[col] == 0).sum()
    nulls = df[col].isna().sum()
    print(f"{col}: zeros={zeros}, nulls={nulls} -> used in {feat}")

**Conclusion:** Any zero in these columns would cause **division by zero**. The cleaning step is to **replace 0 with NaN** in the denominator before dividing, so the ratio becomes NaN instead of inf. This is what `feature_engineering.clean_application` does (e.g. `df["AMT_CREDIT"].replace(0, np.nan)`).

## 4. Apply the same cleaning as `clean_application` and compare

Replicating the logic in `src/feature_engineering.clean_application` to verify it matches our EDA.

In [ ]:
def clean_application_eda(df: pd.DataFrame) -> pd.DataFrame:
    """Same cleaning as src/feature_engineering.clean_application."""
    df = df.copy()
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).round(1)
    df["EMPLOYMENT_YEARS"] = (-df["DAYS_EMPLOYED"] / 365.25).round(1)
    df["INCOME_TO_CREDIT_RATIO"] = df["AMT_INCOME_TOTAL"] / df["AMT_CREDIT"].replace(0, np.nan)
    df["ANNUITY_TO_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    df["CREDIT_TO_GOODS_RATIO"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)
    df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    return df

df_clean = clean_application_eda(df)
new_cols = ["AGE_YEARS", "EMPLOYMENT_YEARS", "INCOME_TO_CREDIT_RATIO", "ANNUITY_TO_INCOME_RATIO", "CREDIT_TO_GOODS_RATIO", "INCOME_PER_FAMILY_MEMBER"]
print("Derived features (summary after cleaning):")
df_clean[new_cols].describe()

In [ ]:
print("EMPLOYMENT_YEARS after replacing 365243 (no more ~1000 years):")
print(df_clean["EMPLOYMENT_YEARS"].describe())
print("\nNo inf in ratios (division-by-zero -> NaN):")
for c in ["INCOME_TO_CREDIT_RATIO", "ANNUITY_TO_INCOME_RATIO", "CREDIT_TO_GOODS_RATIO", "INCOME_PER_FAMILY_MEMBER"]:
    print(f"  {c}: inf count = {np.isinf(df_clean[c]).sum()}")

## 5. Summary: EDA → cleaning steps in `feature_engineering`

| EDA finding | Cleaning step in `clean_application()` |
|-------------|----------------------------------------|
| `DAYS_EMPLOYED == 365243` is a sentinel for unemployed/unknown | `DAYS_EMPLOYED.replace(365243, np.nan)` |
| Need age in years | `AGE_YEARS = (-DAYS_BIRTH / 365.25).round(1)` |
| Need employment length in years (after fixing sentinel) | `EMPLOYMENT_YEARS = (-DAYS_EMPLOYED / 365.25).round(1)` |
| Ratios useful for risk; denominators can be 0 | Divide by `col.replace(0, np.nan)` for AMT_CREDIT, AMT_INCOME_TOTAL, AMT_GOODS_PRICE, CNT_FAM_MEMBERS |

These are exactly the cleaning and derived-feature steps implemented in `src/feature_engineering.clean_application`.